# Lab 4 c.d.

Autor: Hubert Kozierkiewicz

Druga część, żeby zdjąć stare kontenery i postawić nowe

## Milvus

Połączenie

In [28]:
from pymilvus import MilvusClient

host = "localhost"
port = "19530"

milvus_client = MilvusClient(
    host=host,
    port=port
)


Schema

In [29]:
from pymilvus import FieldSchema, DataType, CollectionSchema

VECTOR_LENGTH = 768  # check the dimensionality for Silver Retriever Base (v1.1) model

id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, description="Primary id")
text = FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=4096, description="Page text")
embedding_text = FieldSchema("embedding", dtype=DataType.FLOAT_VECTOR, dim=VECTOR_LENGTH, description="Embedded text")

fields = [id_field, text, embedding_text]

schema = CollectionSchema(fields=fields, auto_id=True, enable_dynamic_field=True, description="RAG Texts collection")


Collection

In [30]:
COLLECTION_NAME = "rag_texts_and_embeddings"

milvus_client.create_collection(
    collection_name=COLLECTION_NAME,
    schema=schema
)

index_params = milvus_client.prepare_index_params()

index_params.add_index(
    field_name="embedding", 
    index_type="HNSW",
    metric_type="L2",
    params={"M": 4, "efConstruction": 64}  # lower values for speed
) 

milvus_client.create_index(
    collection_name=COLLECTION_NAME,
    index_params=index_params
)

# checkout our collection
print(milvus_client.list_collections())

# describe our collection
print(milvus_client.describe_collection(COLLECTION_NAME))


['rag_texts_and_embeddings']
{'collection_name': 'rag_texts_and_embeddings', 'auto_id': True, 'num_shards': 1, 'description': 'RAG Texts collection', 'fields': [{'field_id': 100, 'name': 'id', 'description': 'Primary id', 'type': <DataType.INT64: 5>, 'params': {}, 'auto_id': True, 'is_primary': True}, {'field_id': 101, 'name': 'text', 'description': 'Page text', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'field_id': 102, 'name': 'embedding', 'description': 'Embedded text', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'functions': [], 'aliases': [], 'collection_id': 470236435386729567, 'consistency_level': 2, 'properties': {'timezone': 'UTC'}, 'num_partitions': 1, 'enable_dynamic_field': True, 'created_timestamp': 470236441256394770, 'update_timestamp': 470236441256394770}


Dodanie dokumentów

In [31]:
# define data source and destination
## the document origin destination from which document will be downloaded 
pdf_url = "https://www.iab.org.pl/wp-content/uploads/2024/04/Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the document
file_name = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.pdf"

## local destination of the processed document 
file_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska.json"

## local destination of the embedded pages of the document
embeddings_json = "Przewodnik-po-sztucznej-inteligencji-2024_IAB-Polska-Embeddings.json"

## local destination of all above local required files
data_dir = "./data"


In [32]:
# download data
import os
import requests

def download_pdf_data(pdf_url: str, file_name: str) -> None:
    response = requests.get(pdf_url, stream=True)
    os.makedirs(data_dir, exist_ok=True)
    with open(os.path.join(data_dir, file_name), "wb") as file:
        for block in response.iter_content(chunk_size=1024):
            if block:
                file.write(block)

download_pdf_data(pdf_url, file_name)


Paginacja i chunkowanie

In [33]:
# prepare data

import fitz
import json


def extract_pdf_text(file_name, file_json):
    document = fitz.open(os.path.join(data_dir, file_name))
    pages = []

    for page_num in range(len(document)):
        page = document.load_page(page_num)
        page_text = page.get_text()
        pages.append({"page_num": page_num, "text": page_text})

    with open(os.path.join(data_dir, file_json), "w", encoding="utf-8") as file:
        json.dump(pages, file, indent=4, ensure_ascii=False)

extract_pdf_text(file_name, file_json)


Wektoryzacja

In [34]:
# vectorize data

import torch
import numpy as np
from sentence_transformers import SentenceTransformer


def generate_embeddings(file_json, embeddings_json, model):
    pages = []
    with open(os.path.join(data_dir, file_json), "r", encoding="utf-8") as file:
        data = json.load(file)

    for page in data:
        pages.append(page["text"])

    embeddings = model.encode(pages)

    embeddings_paginated = []
    for page_num in range(len(embeddings)):
        embeddings_paginated.append({"page_num": page_num, "embedding": embeddings[page_num].tolist()})

    with open(os.path.join(data_dir, embeddings_json), "w", encoding="utf-8") as file:
        json.dump(embeddings_paginated, file, indent=4, ensure_ascii=False)

model_name = "ipipan/silver-retriever-base-v1.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(model_name, device=device)
generate_embeddings(file_json, embeddings_json, model)


Ingest

In [35]:
def insert_embeddings(file_json, embeddings_json, client=milvus_client):
    rows = []
    with open(os.path.join(data_dir, file_json), "r", encoding="utf-8") as t_f, open(os.path.join(data_dir, embeddings_json), "r", encoding="utf-8") as e_f:
        text_data, embedding_data = json.load(t_f), json.load(e_f)
        text_data =  list(map(lambda d: d["text"], text_data))
        embedding_data = list(map(lambda d: d["embedding"], embedding_data))
        
        for page, (text, embedding) in enumerate(zip(text_data, embedding_data)):
            rows.append({"text":text, "embedding": embedding})

    client.insert(collection_name="rag_texts_and_embeddings", data=rows)


insert_embeddings(file_json, embeddings_json)

# load inserted data into memory
milvus_client.load_collection("rag_texts_and_embeddings")


Retreival

In [36]:
# search

def search(model, query, client=milvus_client, limit=1):
    embedded_query = model.encode(query).tolist()
    result = client.search(
        collection_name="rag_texts_and_embeddings", 
        data=[embedded_query], 
        limit=limit,
        search_params={"metric_type": "L2"},
        output_fields=["text"]
    )
    return result


result = search(model, query="Czym jest sztuczna inteligencja")

print(result[0][0]['entity']['text'])


Historia powstania
sztucznej inteligencji
7
W języku potocznym „sztuczny" oznacza to, co
jest 
wytworem 
mającym 
naśladować 
coś
naturalnego. W takim znaczeniu używamy
terminu ,,sztuczny'', gdy mówimy o sztucznym
lodowisku lub oku. Sztuczna inteligencja byłaby
czymś (programem, maszyną) symulującym
inteligencję naturalną, ludzką.
Sztuczna inteligencja (AI) to obszar informatyki,
który skupia się na tworzeniu programów
komputerowych zdolnych do wykonywania
zadań, które wymagają ludzkiej inteligencji. 
Te zadania obejmują rozpoznawanie wzorców,
rozumienie języka naturalnego, podejmowanie
decyzji, uczenie się, planowanie i wiele innych.
Głównym celem AI jest stworzenie systemów,
które są zdolne do myślenia i podejmowania
decyzji na sposób przypominający ludzki.
Historia sztucznej inteligencji sięga lat 50. 
XX wieku, kiedy to powstały pierwsze koncepcje
i modele tego, co mogłoby stać się sztuczną
inteligencją. Jednym z pionierów był Alan
Turing, który sformułował test Turinga, mający
na 

## Openai API

Zmieniłem na openai, bo nie mam kredytów w gemini a nie mogę spowrotem przełączyć na free tier

In [37]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

OPENAI_KEY = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI(api_key=OPENAI_KEY)

MODEL = "gpt-6-luna"

def generate_response(prompt: str):
    try:
        response = openai_client.responses.create(
            model=MODEL,
            input=prompt,
        )
        return response.output_text
    except Exception as e:
        print(f"Error generating response: {e}")
        return None


In [58]:
def build_prompt(context: str, query: str, style: str = '') -> str:
    prompt = f"""Odpowiedz na pytanie tylko na podstawie kontekstu.
    Zastosuj podany poniżej styl, o ile został dodany.
    Jeśli w kontekście nie ma odpowiedzi, napisz "Brak informacji".

    Styl:
    {style}
    
    Kontekst:
    {context}
    
    Pytanie:
    {query}
    """
    return prompt
    

def rag(model, query: str, style: str = "", limit: int = 5) -> str:
    # having all prepared functions, you can combine them together and try to build your own RAG!
    result = search(model, query, limit=limit)
    context = "\n\n".join(hit["entity"]["text"] for hit in result[0])
    prompt = build_prompt(context, query, style)
    response = generate_response(prompt)
    return response

## Testy

In [59]:
# styl
print(rag(model, "Czym jest sztuczna inteligencja?", style="Mów jak silnie brainrotten gen alpha z uzyciem wstawek i slangu jak rizz, ohio, gyatt itd"))

Sztuczna inteligencja to obszar informatyki zajmujący się tworzeniem programów i maszyn, które naśladują ludzką inteligencję, wykonując zadania takie jak uczenie się, rozpoznawanie wzorców, rozumienie języka i podejmowanie decyzji. Totalny rizz technologii, bez ohio.


In [53]:
# jezyk
print(rag(model, "Czym jest sztuczna inteligencja?", style="Answer in English, despite the question being asked in Polish"))

Artificial intelligence (AI) is a field of computer science focused on creating programs or machines that can perform tasks requiring human intelligence, such as recognizing patterns, understanding language, making decisions, learning, and planning.


In [60]:
# format
print(rag(model, "Czym jest sztuczna inteligencja?", style="Odpowiedz w 3 punktach, dodaj emoji (dużo crying emoji, też w środku tekstu)"))

- Sztuczna inteligencja (SI) to obszar informatyki zajmujący się tworzeniem programów i maszyn naśladujących ludzką inteligencję 😭😭.
- Systemy SI wykonują zadania takie jak rozpoznawanie wzorców, rozumienie języka, uczenie się, planowanie i podejmowanie decyzji 😭.
- Ich celem jest działanie w sposób przypominający ludzkie myślenie, choć współczesna SI jest zwykle inteligentna tylko w ograniczonym zakresie 😭😭😭.


In [55]:
# persona
print(rag(model, "Czym jest sztuczna inteligencja?", style="Mów jak głęboko wierzący góral z silną gwarą, który tłumaczy wszystko wolą boską"))

Sztuczna inteligencja, juhasie, to dziedzina informatyki, co tworzy programy i maszyny naśladujące ludzką inteligencję. Umieją wykonywać zadania takie jak rozpoznawanie wzorców, rozumienie języka, uczenie się czy podejmowanie decyzji — choć, jak Pan Bóg dał, człowiekowi rozum, tak AI działa ino w wyznaczonym zakresie.


In [56]:
# halucynacje - czegos takiego nie ma w pdfie
print(rag(model, "Kiedy HalfLife 3"))

Brak informacji
